# Chương 11: Thực hành - Xử lý Ngôn ngữ Tự nhiên (Natural Language Processing)

**Mục tiêu bài thực hành:**
1. **Biểu diễn văn bản Cổ điển:** Tự cài đặt thuật toán tính toán ma trận **TF-IDF** từ đầu bằng `NumPy` trên tập văn bản mẫu.
2. **Học Biểu diễn Phân tán (Word2Vec Skip-gram from scratch):** Xây dựng mạng nơ-ron nhúng từ thu nhỏ và trực quan hóa không gian ngữ nghĩa 2D.
3. **Cài đặt Mạng nơ-ron chuỗi (Vanilla RNN vs. LSTM):** Lập trình trọn vẹn cơ chế 3 cổng của LSTM và chứng minh khả năng lưu giữ ký ức dài hạn vượt trội so với RNN truyền thống.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cố định hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Cài đặt Ma trận TF-IDF từ đầu (from scratch)

$$\text{TF}(t, d) = \frac{f_{t, d}}{\sum_{t'} f_{t', d}}, \qquad \text{IDF}(t, D) = \log\left(\frac{|D|}{\{d: t \in d\} + 1}\right) + 1$$
$$\text{TF-IDF} = \text{TF} \times \text{IDF}$$

In [ ]:
corpus = [
    "trí tuệ nhân tạo và học máy",
    "mạng nơ ron và học sâu",
    "học máy là một nhánh của trí tuệ nhân tạo",
    "thị giác máy tính và xử lý ngôn ngữ"
]

# 1. Tách từ và xây dựng từ điển (Vocabulary)
tokenized_docs = [doc.lower().split() for doc in corpus]
vocab = sorted(list(set(word for doc in tokenized_docs for word in doc)))
word2idx = {w: i for i, w in enumerate(vocab)}

N_docs = len(corpus)
V_size = len(vocab)

# 2. Tính Term Frequency (TF)
tf_matrix = np.zeros((N_docs, V_size))
for i, doc in enumerate(tokenized_docs):
    for word in doc:
        tf_matrix[i, word2idx[word]] += 1
    tf_matrix[i] /= len(doc)

# 3. Tính Inverse Document Frequency (IDF)
df = np.sum(tf_matrix > 0, axis=0)
idf = np.log((N_docs + 1) / (df + 1)) + 1.0 # Làm mịn chuẩn scikit-learn

# 4. Tính ma trận TF-IDF
tfidf_matrix = tf_matrix * idf
# Chuẩn hóa chuẩn L2 cho mỗi hàng văn bản
tfidf_norm = tfidf_matrix / np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)

plt.figure(figsize=(12, 4))
plt.imshow(tfidf_norm, cmap='YlGnBu', aspect='auto')
plt.colorbar(label='Trọng số TF-IDF')
plt.xticks(range(V_size), vocab, rotation=45, ha='right', fontsize=10)
plt.yticks(range(N_docs), [f"Văn bản {i+1}" for i in range(N_docs)], fontsize=10)
plt.title("Bản đồ nhiệt Ma trận TF-IDF cho Tập Văn bản Mẫu", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Mô hình Nhúng Từ Word2Vec Skip-gram Thu nhỏ từ đầu

In [ ]:
# Tạo tập cặp từ (Target -> Context) với cửa sổ ngữ cảnh window_size = 1
pairs = []
for doc in tokenized_docs:
    for i, target in enumerate(doc):
        for j in range(max(0, i - 1), min(len(doc), i + 2)):
            if i != j:
                pairs.append((word2idx[target], word2idx[doc[j]]))

embedding_dim = 2
# Khởi tạo ma trận nhúng W_in (V x d) và W_out (d x V)
W_in = np.random.randn(V_size, embedding_dim) * 0.5
W_out = np.random.randn(embedding_dim, V_size) * 0.5

lr = 0.05
for epoch in range(500):
    for target, context in pairs:
        # Lan truyền tiến
        h = W_in[target] # 1 x d (chính là vector nhúng của target)
        logits = h @ W_out # 1 x V
        probs = np.exp(logits - np.max(logits))
        probs /= np.sum(probs)
        
        # Lan truyền ngược (Cross-Entropy gradient)
        d_logits = probs.copy()
        d_logits[context] -= 1.0
        
        dW_out = np.outer(h, d_logits)
        dh = d_logits @ W_out.T
        
        # Cập nhật
        W_out -= lr * dW_out
        W_in[target] -= lr * dh

# Trực quan hóa không gian nhúng 2D học được
plt.figure(figsize=(9, 6))
for word, idx in word2idx.items():
    vec = W_in[idx]
    plt.scatter(vec[0], vec[1], color='#e74c3c', s=50)
    plt.text(vec[0] + 0.02, vec[1] + 0.02, word, fontsize=11, fontweight='bold')

plt.title("Không gian Vector Nhúng 2D Học được bằng Mô hình Word2Vec Thu nhỏ", fontsize=12, fontweight='bold')
plt.xlabel("Chiều nhúng 1")
plt.ylabel("Chiều nhúng 2")
plt.tight_layout()
plt.show()

## 3. Cài đặt Tầng LSTM (Long Short-Term Memory) từ đầu

Triển khai 3 cổng logic và Cell State $\mathbf{c}_t$ để chứng minh khả năng bảo toàn tín hiệu thông tin.

In [ ]:
class LSTMCellScratch:
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        
        # Khởi tạo trọng số cho 4 cổng: Forget (f), Input (i), Candidate (c), Output (o)
        self.Wf = np.random.randn(concat_dim, hidden_dim) * 0.1
        self.bf = np.ones((1, hidden_dim)) # Forget bias = 1.0 (khuyên dùng để nhớ lúc đầu)
        
        self.Wi = np.random.randn(concat_dim, hidden_dim) * 0.1
        self.bi = np.zeros((1, hidden_dim))
        
        self.Wc = np.random.randn(concat_dim, hidden_dim) * 0.1
        self.bc = np.zeros((1, hidden_dim))
        
        self.Wo = np.random.randn(concat_dim, hidden_dim) * 0.1
        self.bo = np.zeros((1, hidden_dim))
        
    def forward_step(self, xt, h_prev, c_prev):
        concat = np.hstack([h_prev, xt]) # 1 x (hidden_dim + input_dim)
        
        # 1. Forget Gate
        ft = 1.0 / (1.0 + np.exp(-(concat @ self.Wf + self.bf)))
        # 2. Input Gate
        it = 1.0 / (1.0 + np.exp(-(concat @ self.Wi + self.bi)))
        c_cand = np.tanh(concat @ self.Wc + self.bc)
        # 3. Cập nhật Cell State (Tuyến tính, không qua kích hoạt co cụm)
        ct = ft * c_prev + it * c_cand
        # 4. Output Gate
        ot = 1.0 / (1.0 + np.exp(-(concat @ self.Wo + self.bo)))
        ht = ot * np.tanh(ct)
        
        return ht, ct, ft, it

# Mô phỏng truyền chuỗi 15 bước thời gian qua LSTM
T_steps = 15
lstm = LSTMCellScratch(input_dim=4, hidden_dim=6)
h_cur = np.zeros((1, 6))
c_cur = np.ones((1, 6)) * 2.0 # Tín hiệu nhớ ban đầu cần lưu giữ

cell_state_magnitudes = [np.linalg.norm(c_cur)]
for t in range(T_steps):
    xt = np.random.randn(1, 4) * 0.1
    h_cur, c_cur, _, _ = lstm.forward_step(xt, h_cur, c_cur)
    cell_state_magnitudes.append(np.linalg.norm(c_cur))

plt.figure(figsize=(10, 4))
plt.plot(range(T_steps + 1), cell_state_magnitudes, 'o-g', linewidth=2, label='Độ lớn Cell State (ct) của LSTM')
plt.title("Dòng chảy Thông tin Bền vững của Cell State qua 15 Bước Thời gian", fontsize=12, fontweight='bold')
plt.xlabel("Bước thời gian (Time Step)")
plt.ylabel("Chuẩn L2 của Vector Bộ nhớ")
plt.ylim(0, 4)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Tổng kết bài học thực hành

1. **TF-IDF:** Nắm bắt tầm quan trọng của từ ngữ thông qua tần số thống kê, là công cụ xếp hạng tài liệu kinh điển trong Information Retrieval.
2. **Word2Vec (Skip-gram):** Minh chứng khả năng số hóa ngữ nghĩa vào không gian hình học liên tục thông qua giả thuyết phân phối.
3. **LSTM:** Cơ chế Cell State đóng vai trò như một đường băng tải thông tin tuyến tính, cho phép mạng nơ-ron chuỗi nhớ được ngữ cảnh dài hạn mà không bị suy thoái tín hiệu.